# F1 INSIGHT — 02 · OLAP Analysis

Hands-on OLAP against the star schema using plain SQL **exactly the way the
`/api/olap/*` endpoints compile them** (rollup, drilldown, slice, dice, pivot).

> Requires a populated warehouse (see notebook 01 / `docs/setup.md`).

## Setup + helpers

`q(sql, **params)` runs parameterised SQL through the same engine the API uses.

In [ ]:
# Convenience setup: make the backend package importable (run from repo root).
import os, sys
ROOT = os.getcwd()
for candidate in (os.path.join(ROOT, "backend"), os.path.join(ROOT, "..", "backend")):
    if os.path.isdir(candidate):
        sys.path.insert(0, candidate)
        break
import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("pandas", pd.__version__)


from sqlalchemy import text
from app.db import SessionLocal


def q(sql, **params):
    "Run SQL against the warehouse and return rows as dicts."
    with SessionLocal() as db:
        rows = db.execute(text(sql), params).mappings().all()
    return [dict(r) for r in rows]


def counts():
    tables = ["dim_date", "dim_driver", "dim_constructor", "dim_circuit", "dim_race",
              "fact_race_result", "fact_qualifying", "fact_lap", "fact_pit_stop",
              "fact_driver_standing", "fact_constructor_standing"]
    out = []
    total = 0
    for t in tables:
        n = q(f"SELECT COUNT(*) AS n FROM {t}")[0]["n"]
        total += n
        out.append((t, n))
    out.append(("TOTAL", total))
    return out

## ROLL-UP — aggregate up a hierarchy

Group race results by `constructor` then by `season` — the classic dimensions
of an F1 cube, with standard measures (races, wins, podiums, points, avg
finish, DNF rate, avg position gain).

In [ ]:
rollup = pd.DataFrame(q(
    "SELECT c.name AS constructor, f.season, "
    "       COUNT(*) AS races, SUM(f.position = 1) AS wins, "
    "       SUM(f.position <= 3) AS podiums, ROUND(SUM(f.points), 2) AS points, "
    "       ROUND(AVG(CASE WHEN f.position IS NOT NULL THEN f.position END), 2) AS avg_finish, "
    "       ROUND(SUM(f.position IS NULL) / COUNT(*), 3) AS dnf_rate, "
    "       ROUND(AVG(f.position_gain), 2) AS avg_position_gain "
    "FROM fact_race_result f "
    "JOIN dim_constructor c ON c.constructor_id = f.constructor_id "
    "WHERE f.season >= 2020 "
    "GROUP BY c.name, f.season "
    "ORDER BY points DESC LIMIT 15"
))
rollup

## DRILL-DOWN — season → race → driver

Descend the hierarchy: pick the most recent season, look at its races, then
drill into one race's driver results.

In [ ]:
races_recent = pd.DataFrame(q(
    "SELECT r.race_id, CONCAT(r.season, ' R', r.round, ' - ', r.name) AS race, "
    "       r.date, ci.name AS circuit, COUNT(*) AS entries, "
    "       (SELECT d2.full_name FROM fact_race_result fw "
    "        JOIN dim_driver d2 ON d2.driver_id = fw.driver_id "
    "        WHERE fw.race_id = r.race_id AND fw.position = 1) AS winner "
    "FROM fact_race_result f "
    "JOIN dim_race r ON r.race_id = f.race_id "
    "JOIN dim_circuit ci ON ci.circuit_id = f.circuit_id "
    "WHERE r.season = (SELECT MAX(season) FROM dim_race) "
    "GROUP BY r.race_id, r.season, r.round, r.date, r.name, ci.name "
    "ORDER BY r.round LIMIT 24"
))
races_recent

Now drill into a single race down to the driver level:

In [ ]:
race_id = int(races_recent.iloc[0]["race_id"])
pd.DataFrame(q(
    "SELECT p.grid, p.position, p.points, d.full_name AS driver, "
    "       c.name AS constructor, p.position_gain, p.status "
    "FROM fact_race_result p "
    "JOIN dim_driver d ON d.driver_id = p.driver_id "
    "JOIN dim_constructor c ON c.constructor_id = p.constructor_id "
    "WHERE p.race_id = :race_id ORDER BY COALESCE(p.position, 999)"
, race_id=race_id))

## SLICE — fix one dimension

Slice the cube on a single dimension value (a constructor) and break it down
by season.

In [ ]:
pd.DataFrame(q(
    "SELECT f.season, COUNT(*) AS entries, SUM(f.position = 1) AS wins, "
    "       ROUND(SUM(f.points), 2) AS points, "
    "       ROUND(AVG(CASE WHEN f.position IS NOT NULL THEN f.position END), 2) AS avg_finish "
    "FROM fact_race_result f "
    "JOIN dim_constructor c ON c.constructor_id = f.constructor_id "
    "WHERE c.name = 'Ferrari' "
    "GROUP BY f.season ORDER BY f.season DESC LIMIT 10"
))

## DICE — filter on multiple dimensions

Keep the sub-cube of two specified constructors over a season range and
aggregate it.

In [ ]:
pd.DataFrame(q(
    "SELECT f.season, c.name AS constructor, COUNT(*) AS entries, "
    "       SUM(f.position <= 3) AS podiums, ROUND(SUM(f.points), 2) AS points "
    "FROM fact_race_result f "
    "JOIN dim_constructor c ON c.constructor_id = f.constructor_id "
    "WHERE f.season BETWEEN 2021 AND 2024 "
    "  AND c.name IN ('Red Bull', 'Mercedes') "
    "GROUP BY f.season, c.name ORDER BY f.season, points DESC"
))

## PIVOT — cross-tabulate two dimensions

Rebuild the `rows=constructor × cols=season × measure=points` pivot the API
serves — pandas gives the same result for exploration.

In [ ]:
pivot = (
    pd.DataFrame(q(
        "SELECT c.name AS constructor, f.season, SUM(f.points) AS points "
        "FROM fact_race_result f "
        "JOIN dim_constructor c ON c.constructor_id = f.constructor_id "
        "WHERE f.season BETWEEN 2018 AND 2025 "
        "GROUP BY c.name, f.season"
    ))
    .pivot_table(index="constructor", columns="season", values="points",
                 aggfunc="sum", fill_value=0)
    .sort_index(axis=1, ascending=True)
)
pivot.sort_values(pivot.columns[-1], ascending=False).head(8)

Pivot as a heatmap:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

top = pivot.sort_values(pivot.columns[-1], ascending=False).head(8)
data = top.to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(data, cmap="Reds")
ax.set_xticks(range(len(pivot.columns)), pivot.columns)
ax.set_yticks(range(len(top.index)), top.index)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j, i, f"{data[i, j]:.0f}", ha="center", va="center",
                color="white" if data[i, j] > 200 else "black")
ax.set_title("Constructor points - season by season (top 8)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()